# Attempt 003: prompted judge runtime and memory probe

Build a pinned llama.cpp binary for Kaggle T4 GPUs, then test official 4-bit Gemma
31B and 26B-A4B checkpoints. No model training or submission. This setup notebook
uses internet to retrieve source; downstream notebooks reuse its compiled runtime
with internet disabled. Weight files stay in attached model inputs.


In [ ]:
import os,json,sys
from pathlib import Path
WORK=Path('/kaggle/working')
MODULES=WORK/'bundled_code/arena';MODULES.mkdir(parents=True,exist_ok=True)
(MODULES/'__init__.py').write_text('')
(MODULES/'judge_runtime.py').write_text('"""Compile a pinned llama.cpp runtime and probe official Gemma GGUF models on Kaggle."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport tarfile\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nimport requests\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2) + \'\\n\')\n\n\ndef runtime_environment(executable):\n    environment = os.environ.copy()\n    paths = [str(Path(executable).parent), \'/usr/local/cuda/lib64\', \'/usr/local/cuda/targets/x86_64-linux/lib\']\n    environment[\'LD_LIBRARY_PATH\'] = \':\'.join(paths) + \':\' + environment.get(\'LD_LIBRARY_PATH\', \'\')\n    return environment\n\n\ndef install_runtime(distribution, out):\n    """Install an official release asset only after checking its published digest."""\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    root = Path(\'/tmp/arena_llama_binary\')\n    root.mkdir(exist_ok=True)\n    archive = root / \'runtime.tar.gz\'\n    print(\'RUNTIME_BINARY\', json.dumps(distribution), flush=True)\n    with urllib.request.urlopen(distribution[\'url\'], timeout=120) as response:\n        with archive.open(\'wb\') as stream: shutil.copyfileobj(response, stream)\n    assert hashlib.sha256(archive.read_bytes()).hexdigest() == distribution[\'sha256\'], \'Release digest mismatch\'\n    with tarfile.open(archive) as tar: tar.extractall(root, filter=\'data\')\n    executable = next(root.rglob(\'llama-server\'))\n    binary = out / \'bin\'\n    binary.mkdir(exist_ok=True)\n    for path in executable.parent.iterdir():\n        if path.name == \'llama-server\' or \'.so\' in path.name:\n            shutil.copy2(path, binary / path.name, follow_symlinks=True)\n    executable = binary / \'llama-server\'\n    executable.chmod(0o755)\n    info = {**distribution, \'version\': subprocess.check_output([str(executable), \'--version\'],\n        stderr=subprocess.STDOUT, text=True, env=runtime_environment(executable)).strip()}\n    write_json(out / \'runtime.json\', info)\n    return executable\n\n\ndef build_runtime(revision, out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    print(\'RUNTIME_SETUP\', json.dumps({\'revision\': revision}), flush=True)\n    root = Path(\'/tmp/arena_llama_build\')\n    root.mkdir(exist_ok=True)\n    archive = root / \'source.tar.gz\'\n    with urllib.request.urlopen(f\'https://github.com/ggml-org/llama.cpp/archive/{revision}.tar.gz\', timeout=120) as response:\n        with archive.open(\'wb\') as stream:\n            shutil.copyfileobj(response, stream)\n    digest = hashlib.sha256(archive.read_bytes()).hexdigest()\n    with tarfile.open(archive) as tar:\n        tar.extractall(root, filter=\'data\')\n    source = root / (\'llama.cpp-\' + revision)\n    build = root / \'build\'\n    # Kaggle exposes libcuda.so.1 without the unversioned developer symlink that\n    # FindCUDAToolkit expects. Supply the actual installed driver library.\n    libraries = subprocess.check_output([\'ldconfig\', \'-p\'], text=True)\n    drivers = re.findall(r\'libcuda\\.so(?:\\.1)?\\s+\\([^\\n]+\\)\\s+=>\\s+(\\S+)\', libraries)\n    drivers += [str(p) for directory in [\'/usr/lib64-nvidia\', \'/usr/lib/x86_64-linux-gnu\', \'/usr/local/nvidia/lib64\']\n                for p in Path(directory).glob(\'libcuda.so*\')]\n    driver = next((p for p in drivers if Path(p).exists()), None)\n    assert driver, \'The allocated GPU driver library is not visible\'\n    print(\'CUDA_DRIVER\', driver, flush=True)\n    commands = [\n        [\'cmake\', \'-S\', str(source), \'-B\', str(build), \'-DGGML_CUDA=ON\',\n         \'-DCMAKE_CUDA_ARCHITECTURES=75\', \'-DCMAKE_BUILD_TYPE=Release\',\n         \'-DCUDA_cuda_driver_LIBRARY=\' + driver,\n         \'-DLLAMA_CURL=OFF\', \'-DLLAMA_BUILD_TESTS=OFF\', \'-DLLAMA_BUILD_EXAMPLES=OFF\',\n         \'-DLLAMA_BUILD_SERVER=ON\', \'-DBUILD_SHARED_LIBS=ON\'],\n        [\'cmake\', \'--build\', str(build), \'--config\', \'Release\', \'-j\', \'4\', \'--target\', \'llama-server\'],\n    ]\n    started = time.monotonic()\n    with (out / \'build.log\').open(\'w\') as log:\n        for command in commands:\n            print(\'BUILD\', command, flush=True)\n            result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)\n            if result.returncode:\n                print((out / \'build.log\').read_text()[-12000:], flush=True)\n                raise RuntimeError(\'Pinned llama.cpp build failed\')\n    binary = out / \'bin\'\n    binary.mkdir(exist_ok=True)\n    for path in (build / \'bin\').iterdir():\n        if path.name == \'llama-server\' or \'.so\' in path.name:\n            shutil.copy2(path, binary / path.name, follow_symlinks=True)\n    executable = binary / \'llama-server\'\n    assert executable.exists()\n    info = {\'revision\': revision, \'source_sha256\': digest, \'build_seconds\': time.monotonic() - started,\n            \'cuda_architectures\': [75], \'cuda_driver_library\': driver, \'version\': subprocess.check_output([str(executable), \'--version\'],\n                stderr=subprocess.STDOUT, text=True).strip()}\n    write_json(out / \'runtime.json\', info)\n    return executable\n\n\ndef model_file(source):\n    candidates = [Path(\'/kaggle/input/models\') / source, Path(\'/kaggle/input\') / \'/\'.join(source.split(\'/\')[1:])]\n    for root in candidates:\n        if root.exists():\n            files = [p for p in root.rglob(\'*.gguf\') if p.stat().st_size > 1_000_000_000 and \'mmproj\' not in p.name.lower()]\n            if files:\n                return max(files, key=lambda p: p.stat().st_size)\n    raise FileNotFoundError(\'Attached GGUF model not found: \' + source)\n\n\ndef start_server(executable, model, out, *, slots=4, context=2048, port=8080):\n    out = Path(out)\n    log = (out / \'server.log\').open(\'w\')\n    command = [str(executable), \'-m\', str(model), \'--host\', \'127.0.0.1\', \'--port\', str(port),\n               \'-ngl\', \'all\', \'--split-mode\', \'layer\', \'--tensor-split\', \'1,1\',\n               \'--ctx-size\', str(slots * context), \'--parallel\', str(slots),\n               \'--batch-size\', \'2048\', \'--ubatch-size\', \'256\', \'--flash-attn\', \'on\',\n               \'--threads\', \'4\', \'--threads-batch\', \'4\', \'--jinja\']\n    environment = runtime_environment(executable)\n    process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=environment)\n    url = f\'http://127.0.0.1:{port}\'\n    started = time.monotonic()\n    while time.monotonic() - started < 900:\n        if process.poll() is not None:\n            log.close()\n            raise RuntimeError(\'llama-server failed: \' + (out / \'server.log\').read_text()[-12000:])\n        try:\n            if requests.get(url + \'/health\', timeout=2).ok:\n                return process, log, url\n        except requests.RequestException:\n            pass\n        time.sleep(2)\n    process.terminate()\n    log.close()\n    raise TimeoutError(\'Model loading exceeded 15 minutes\')\n\n\ndef post(url, path, payload):\n    response = requests.post(url + path, json=payload, timeout=600)\n    response.raise_for_status()\n    return response.json()\n\n\ndef smoke_probe(executable, source, out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    model = model_file(source)\n    process, log, url = start_server(executable, model, out)\n    try:\n        labels = {text: post(url, \'/tokenize\', {\'content\': text, \'add_special\': False}) for text in [\'A\', \'B\', \'C\']}\n        messages = [{\'role\': \'system\', \'content\': \'Judge two answers. Output A if answer A is better, B if answer B is better, C if tied. Output only one letter.\'},\n                    {\'role\': \'user\', \'content\': \'Question: What is 2+2?\\nAnswer A: 4\\nAnswer B: 5\'}]\n        template = post(url, \'/apply-template\', {\'messages\': messages, \'add_generation_prompt\': True,\n            \'chat_template_kwargs\': {\'enable_thinking\': False}})\n        prompt = template[\'prompt\']\n        thought = \'<|channel>thought\\n<channel|>\'\n        if not prompt.endswith(thought):\n            prompt += thought\n        result = post(url, \'/completion\', {\'prompt\': prompt, \'n_predict\': 1, \'temperature\': 1.0,\n            \'samplers\': [\'temperature\'], \'grammar\': \'root ::= "A" | "B" | "C"\', \'n_probs\': 3,\n            \'post_sampling_probs\': True, \'return_tokens\': True, \'seed\': 20261102})\n        summary = {\'model_source\': source, \'model_filename\': model.name, \'model_bytes\': model.stat().st_size,\n                   \'label_tokens\': labels, \'template\': template, \'response\': result,\n                   \'gpu\': subprocess.check_output([\'nvidia-smi\'], text=True)}\n        write_json(out / \'smoke.json\', summary)\n        print(\'JUDGE_SMOKE\', json.dumps(summary), flush=True)\n    finally:\n        process.terminate()\n        try:\n            process.wait(timeout=30)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        log.close()\n    return summary\n')
sys.path.insert(0,str(MODULES.parent))
from arena.judge_runtime import build_runtime,install_runtime,smoke_probe
executable=install_runtime({'kind': 'official_prebuilt_cuda_12.8', 'tag': 'b11960', 'revision': '8642541b255bfb5dc39cd351a178be09013a55d6', 'url': 'https://github.com/ggml-org/llama.cpp/releases/download/b11960/llama-b11960-bin-ubuntu-cuda-12.8-x64.tar.gz', 'sha256': '64d57beb9abb637c177c6afb313760a0c0782224cac71e74869eed5b8f9a1ea4', 'bytes': 182524002},WORK/"runtime")


In [ ]:
results=[]
for source in ['google/gemma-4/gguf/gemma-4-31b-it-qat-q4_0-gguf/2', 'google/gemma-4/gguf/gemma-4-26b-a4b-it-qat-q4_0-gguf/2']:
    name='gemma31' if '31b' in source else 'gemma26'
    try:
        results.append(smoke_probe(executable,source,WORK/name))
    except Exception as error:
        result={'model_source':source,'error':str(error)}
        (WORK/(name+'_error.json')).write_text(json.dumps(result,indent=2))
        print('JUDGE_PROBE_ERROR',json.dumps(result),flush=True)
        results.append(result)
(WORK/'probe_summary.json').write_text(json.dumps(results,indent=2))
assert any('response' in result for result in results),'No judge model loaded successfully'
